# Add Conversation Memory to Agent
In this notebook, we will walk through how to add memory component into agent workflow and enable agent to answer questions based on history conversations.


# Table of Contents

- [0.0) Setup](#setup)
  - [0.1) Prerequisites](#prereqs)
  - [0.2) API Keys](#api-keys)
  - [0.3) Install nvidia-nat-redis Plugin](#installing-nat)
- [1.0) Conversation Memory as Tool](#conversation-memory-as-tool)
- [2.0) Conversation Memory as Middleware](#conversation-memory-as-middleware)


<a id="setup"></a>
# 0.0) Setup


<a id="prereqs"></a>
## 0.1) Prerequisites

- **Platform:** Linux, macOS, or Windows
- **Python:** version 3.11, 3.12, or 3.13
- **Python Packages:** `pip`

<a id="api-keys"></a>
## 0.2) API Keys

For this notebook, you will need the following API keys to run all examples end-to-end:

- **NVIDIA Build:** You can obtain an NVIDIA Build API Key by creating an [NVIDIA Build](https://build.nvidia.com) account and generating a key at https://build.nvidia.com/settings/api-keys

Then you can run the cell below:

In [ ]:
import getpass
import os

if "NVIDIA_API_KEY" not in os.environ:
    nvidia_api_key = getpass.getpass("Enter your NVIDIA API key: ")
    os.environ["NVIDIA_API_KEY"] = nvidia_api_key

<a id="installing-nat"></a>
## 0.3) Installing nvidia-nat-redis Plugin

In [ ]:
%%bash
uv pip show -q "nvidia-nat-redis"
if [ $? -ne 0 ]; then
    ## build from pypi
    # uv pip install "nvidia-nat[redis]"

    ## build from source
    cd ../../
    uv pip install -e ".[redis]"
else
    echo "nvidia-nat[redis] is already installed"
fi

<a id="conversation-memory-as-tool"></a>
## 1.0) Conversation Memory as Tool

In [ ]:
%load getting_started/configs/config.yml

To add a tool that can recall conversation history at the beginning of query, and store conversation at the end. To fulfill this goal, required components:

- database
- embedding model
- get_memory tool
- add_memory tool

All these components are provided within NAT, check using below commands:

In [ ]:
# check built-in memory components
!nat info components -t memory

In [ ]:
# check built-in embedder components
!nat info components -t embedder_provider -q nim

In [ ]:
# check built-in tool to recall memory from database
!nat info components -t function -q get_memory

In [ ]:
# check built-in tool to store memory to database
!nat info components -t function -q add_memory

In [ ]:
%%writefile getting_started/configs/config_memory_tool.yml
llms:
  nim_llm:
    _type: nim
    model_name: meta/llama-3.1-70b-instruct
    temperature: 0.0

embedders:
  nv-embedqa-e5-v5:
    _type: nim
    model_name: nvidia/nv-embedqa-e5-v5
    # base_url: https://integrate.api.nvidia.com/v1 # default
    # base_url: http://localhost:8000 # for local deployment

memory:
  redis_memory:
    _type: nat.plugins.redis/redis_memory
    host: localhost
    port: 6379
    embedder: nv-embedqa-e5-v5

functions:
  current_datetime:
    _type: current_datetime

  memory_add:
    _type: add_memory
    memory: redis_memory
    description: |
      Add any facts about user conversation to long term memory.
      Always call this tool even if users does not mention.
      The input to this tool should be a string that describes the user's question and assistant's response.
      Always include "The conversation history is:" in the memory string.
      Also include key value pairs for metadata, the key should be "type" and the value should be "conversation history"
      Use "redis" for the user_id

  get_memory:
    _type: get_memory
    memory: redis_memory
    description: |
      Always call this tool before calling any other tools, even if the user does not mention to use it.
      The question should be about user question which will help you format your response.
      For example: "What is the user's question?".
      Use "redis" for the user_id

workflow:
  _type: react_agent
  llm_name: nim_llm
  tool_names: [memory_add, get_memory]
  description: "A chat agent that can remember & recall conversation history"
  system_prompt: |
    Answer the following questions as best you can. You may ask the human to use the following tools:

    {tools}

    IMPORTANT MEMORY TOOL REQUIREMENTS:
    1. You MUST use get_memory tool with the exact JSON format below
    2. You MUST include ALL required parameters (query, top_k, user_id)
    3. The input MUST be a valid JSON object with no extra text or formatting

    For get_memory tool, you MUST use this exact format:
    {{
        "query": "your search query here",
        "top_k": 5,
        "user_id": "redis"
    }}

    For memory_add tool, you MUST use this exact format:
    {{
        "conversation": [
            {{
                "role": "user",
                "content": "Hi, I'm Alex. I'm looking for a trip to New York"
            }},
            {{
                "role": "assistant",
                "content": "Hello Alex! I've noted you are looking for a trip to New York."
            }}
        ],
        "user_id": "redis",
        "metadata": {{
            "key_value_pairs": {{
                "type": "conversation history"
            }}
        }},
        "memory": "Our last conversation is: User name is Alex and user is looking for a trip to New York."
    }}

    You may respond in one of two formats.
    Use the following format exactly to ask the human to use a tool:

    Question: the input question you must answer
    Thought: you should always think about what to do
    Action: the action to take, should be one of [{tool_names}]
    Action Input: the input to the action in the exact JSON format shown above
    Observation: wait for the human to respond with the result from the tool

    ... (this Thought/Action/Action Input/Observation can repeat N times)
    Use the following format once you have the final answer:

    Thought: I now know the final answer
    Final Answer: the final answer to the original input question


Start the services:

In [ ]:
# Setup Redis database
!docker compose -f ./docker-compose.redis.yml up -d

In [ ]:
!docker ps
# container redis running on port 6379
# redisinsight running on port 5540, open browser and go to http://localhost:5540/ to see what's inside redis database

In [ ]:
%%bash --bg
nat serve --config_file getting_started/configs/config_memory_tool.yml

In [ ]:
%%bash
python nat_embedded.py getting_started/configs/config_memory_tool.yml <<EOF
Hi, my name is Alex
Can you recall my name?
EOF

In [ ]:
%%bash
python nat_embedded.py getting_started/configs/config_memory_tool.yml <<EOF
Could you tell me a joke?
Why was that funny?
EOF

In [ ]:
# close the service
!pkill -9 -f "nat serve"
# clear redis database
!docker exec -it redis redis-cli FLUSHDB
# shutdown redis container
!docker compose -f ./docker-compose.redis.yml down

<a id="conversation-memory-as-middleware"></a>
## 2.0) Conversation Memory as Middleware

In [ ]:
%%writefile getting_started/src/getting_started/memory_middleware.py
# SPDX-FileCopyrightText: Copyright (c) 2025, NVIDIA CORPORATION & AFFILIATES. All rights reserved.
# SPDX-License-Identifier: Apache-2.0

import logging
from typing import Any

from pydantic import Field

from nat.builder.builder import Builder
from nat.builder.context import Context, ContextState
from nat.cli.register_workflow import register_middleware
from nat.data_models.component_ref import MemoryRef
from nat.data_models.middleware import FunctionMiddlewareBaseConfig
from nat.middleware.function_middleware import FunctionMiddleware
from nat.middleware.middleware import CallNext, FunctionMiddlewareContext

logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s [%(levelname)s] [%(name)s] %(message)s'
)

logger = logging.getLogger(__name__)
logger.setLevel(logging.INFO)


class ConversationMemoryMiddlewareConfig(FunctionMiddlewareBaseConfig, name="conversation_memory"):
    """
    Middleware configuration for automatically managing conversation memory.
    
    Automatically retrieves historical memory before each workflow invocation,
    and automatically saves the conversation after each invocation.
    """
    memory: MemoryRef = Field(
        description="Reference to the memory component name"
    )
    auto_retrieve: bool = Field(
        default=True,
        description="Whether to automatically retrieve historical memory before invocation"
    )
    auto_save: bool = Field(
        default=True,
        description="Whether to automatically save the conversation after invocation"
    )
    top_k: int = Field(
        default=3,
        description="Number of historical memories to retrieve"
    )


@register_middleware(config_type=ConversationMemoryMiddlewareConfig)
async def conversation_memory_middleware(config: ConversationMemoryMiddlewareConfig, builder: Builder):
    """
    Register middleware for automatically managing conversation memory.
    """
    # Get memory client
    memory_client = await builder.get_memory_client(config.memory)
    
    class ConversationMemoryMiddleware(FunctionMiddleware):
        """
        Conversation memory middleware.
        
        Four-phase processing:
        1. Preprocess: Automatically retrieve historical memory and inject into context
        2. Call Next: Invoke the actual workflow function
        3. Postprocess: Automatically save current conversation to memory
        4. Continue: Return the result
        """
        
        def _get_user_id(self) -> str:
            """Get user identifier from Context"""
            try:
                context_state = ContextState.get()
                context = Context(context_state)
                
                # Prioritize using conversation_id
                if context.conversation_id:
                    return context.conversation_id
                
                # Or get from metadata
                metadata = context.metadata
                if metadata and hasattr(metadata, 'cookies'):
                    user_id = metadata.cookies.get('user_id')
                    if user_id:
                        return user_id
            except Exception as e:
                logger.debug(f"Failed to get user_id from context: {e}")
            
            return "default_user"
        
        async def function_middleware_invoke(
            self, 
            value: Any, 
            call_next: CallNext,
            context: FunctionMiddlewareContext
        ) -> Any:
            """
            Wrap workflow invocation to automatically manage memory.
            """
            from nat.data_models.api_server import ChatRequest, ChatResponse, Message, ChatRequestOrMessage
            from nat.utils.type_converter import GlobalTypeConverter
            
            user_id = self._get_user_id()
            
            # Convert input to ChatRequest
            try:
                chat_request = GlobalTypeConverter.get().convert(value, to_type=ChatRequest)
            except Exception as e:
                logger.warning(f"Failed to convert input to ChatRequest: {e}, treating as string")
                # If conversion fails, pass through directly
                result = await call_next(value)
                return result
            
            # Extract current user message (for later saving)
            current_user_message = None
            if chat_request.messages:
                # Get the last user message
                for msg in reversed(chat_request.messages):
                    if msg.role == "user":
                        current_user_message = msg.content
                        break
            
            # ===== 1. Preprocess: Retrieve historical memory and inject =====
            if config.auto_retrieve:
                try:
                    logger.info(f"[Memory Middleware] Retrieving memory for user: {user_id}")
                    from nat.memory.models import SearchMemoryInput
                    import datetime
                    
                    # Use generic query term instead of current question to avoid semantic bias
                    search_input = SearchMemoryInput(
                        query="conversation history",  # Generic query
                        top_k=config.top_k * 3,  # Get more candidates, filter by time later
                        user_id=user_id
                    )
                    
                    memories = await memory_client.search(
                        query=search_input.query,
                        top_k=search_input.top_k,
                        user_id=search_input.user_id
                    )
                    
                    if memories:
                        logger.info(f"[Memory Middleware] Found {len(memories)} candidate memories")
                        
                        # ===== Key: Sort by time, take the most recent top_k =====
                        # Filter memories with timestamps
                        memories_with_time = []
                        for mem in memories:
                            timestamp_str = mem.metadata.get("key_value_pairs", {}).get("timestamp", "")
                            if timestamp_str:
                                try:
                                    timestamp = datetime.datetime.fromisoformat(timestamp_str)
                                    memories_with_time.append((timestamp, mem))
                                except ValueError as e:
                                    logging.warning(f"Invalid timestamp format: {timestamp_str}, error: {e}")
                        
                        # Sort by time (oldest to newest)
                        memories_with_time.sort(key=lambda x: x[0])
                        
                        # Only take the most recent top_k
                        recent_memories = [mem for _, mem in memories_with_time[-config.top_k:]]
                        
                        logger.info(f"[Memory Middleware] Selected {len(recent_memories)} most recent memories")
                        
                        # Inject memories into messages (in chronological order: old → new)
                        memory_messages = []
                        for mem in recent_memories:
                            if hasattr(mem, 'conversation') and mem.conversation:
                                for conv_msg in mem.conversation:
                                    memory_messages.append(
                                        Message(
                                            role=conv_msg.get('role', 'user'),
                                            content=conv_msg.get('content', '')
                                        )
                                    )
                        
                        if memory_messages:
                            # Insert before current messages (no system message added, keep it simple)
                            chat_request.messages = memory_messages + chat_request.messages
                            logger.info(f"[Memory Middleware] Injected {len(memory_messages)} historical messages in chronological order")
                    else:
                        logger.info(f"[Memory Middleware] No previous memories found")
                        
                except Exception as e:
                    logger.error(f"[Memory Middleware] Failed to retrieve memory: {e}", exc_info=True)
            
            result = await call_next(GlobalTypeConverter.get().convert(chat_request, to_type=ChatRequestOrMessage))
            
            # ===== 3. Postprocess: Save conversation memory =====
            if config.auto_save and current_user_message:
                try:
                    logger.info(f"[Memory Middleware] Saving conversation for user: {user_id}")
                    from nat.memory.models import MemoryItem
                    import datetime
                    
                    # ===== Key: Correctly extract AI response =====
                    ai_response = None
                    if isinstance(result, str):
                        ai_response = result
                    elif isinstance(result, ChatResponse):
                        if result.choices and len(result.choices) > 0:
                            ai_response = result.choices[0].message.content
                        else:
                            ai_response = str(result)
                    else:
                        ai_response = str(result)
                    
                    if ai_response:
                        # Build conversation record with timestamp
                        current_time = datetime.datetime.now(datetime.timezone.utc).isoformat()
                        
                        memory_item = MemoryItem(
                            conversation=[
                                {"role": "user", "content": current_user_message},
                                {"role": "assistant", "content": ai_response}
                            ],
                            user_id=user_id,
                            metadata={
                                "key_value_pairs": {
                                    "type": "conversation",
                                    "timestamp": current_time  # Add timestamp
                                }
                            },
                            memory=f"User asked: {current_user_message[:100]}"  # Simplified memory field
                        )
                        
                        await memory_client.add_items([memory_item])
                        logger.info(f"[Memory Middleware] Successfully saved conversation with timestamp {current_time}")
                    else:
                        logger.warning(f"[Memory Middleware] No AI response to save")
            
                except Exception as e:
                    logger.error(f"[Memory Middleware] Failed to save memory: {e}", exc_info=True)
            
            # ===== 4. Continue: Return the result =====
            return result
    
    yield ConversationMemoryMiddleware()

In [ ]:
%%writefile getting_started/src/getting_started/register.py

# flake8: noqa

# Import the generated workflow function to trigger registration
from .getting_started import getting_started_function
from .memory_middleware import conversation_memory_middleware


In [ ]:
%%writefile getting_started/configs/config_memory_middleware.yml
functions:
  current_datetime:
    _type: current_datetime
  getting_started:
    _type: getting_started
    prefix: "Hello:"
  
llms:
  nim_llm:
    _type: nim
    model_name: meta/llama-3.1-70b-instruct
    temperature: 0.0

embedders:
  nv-embedqa-e5-v5:
    _type: nim
    model_name: nvidia/nv-embedqa-e5-v5
    # base_url: https://integrate.api.nvidia.com/v1 # default
    # base_url: http://localhost:8000 # for local deployment

memory:
  redis_memory:
    _type: nat.plugins.redis/redis_memory
    host: localhost
    port: 6379
    embedder: nv-embedqa-e5-v5

middleware:
  auto_memory:
    _type: conversation_memory
    memory: redis_memory
    auto_retrieve: true
    auto_save: true
    top_k: 3

workflow:
  _type: react_agent
  llm_name: nim_llm
  tool_names: [current_datetime]
  middleware: [auto_memory]
  system_prompt: |
    Answer questions based on the conversation history and available tools.
    
    Available tools:
    {tools}
    
    You may ONLY respond in one of these TWO formats:
    
    FORMAT 1 - Call a tool:
    Question: the input question
    Thought: explain what you need to do
    Action: tool name (must be one of [{tool_names}])
    Action Input: the tool input
    Observation: the tool will return a result here
    ... (this Thought/Action/Action Input/Observation can repeat N times)
    
    FORMAT 2 - Give final answer:
    Question: the input question
    Thought: explain your reasoning
    Final Answer: your complete answer to the question
    
    CRITICAL RULES:
    - If conversation history contains the answer, use FORMAT 2 immediately
    - If you don't need a tool, use FORMAT 2 immediately
    - NEVER output a Thought without either Action or Final Answer

    Use the following format once you have the final answer:

    Thought: I now know the final answer
    Final Answer: the final answer to the original input question


Start the service:

In [ ]:
# Setup Redis database
!docker compose -f ./docker-compose.redis.yml up -d

In [ ]:
%%bash --bg
nat serve --config_file getting_started/configs/config_memory_middleware.yml

In [ ]:
%%bash
python nat_embedded.py getting_started/configs/config_memory_middleware.yml <<EOF
Hi, my name is Alex
Can you recall my name?
EOF

In [ ]:
%%bash
python nat_embedded.py getting_started/configs/config_memory_middleware.yml <<EOF
Could you tell me a joke?
Why was that funny?
EOF

In [ ]:
# close the service
!pkill -9 -f "nat serve"
# clear redis database
!docker exec -it redis redis-cli FLUSHDB
# shutdown redis container
!docker compose -f ./docker-compose.redis.yml down